# 12 动态 batch 如何按 token budget 设计？

## 面试回答主线

动态 batch 的目标不是固定“每卡多少条”，而是在显存、最大长度、packing 规则和延迟约束下尽可能填满 token budget。对于变长对话，固定样本数 batch 会制造大量 padding；长度分桶、贪心装箱或按 token 数累积能提升利用率。面试要说明训练动态 batch 与在线 serving dynamic batching 的约束不同：后者还受 deadline 和 KV cache 影响。实验将六条客服请求装入每批最多 10 token 的训练 microbatch，比较固定三条一批与长度感知贪心装箱。

**核心公式：** 固定条数 batch 的 padding 利用率可写为 $\sum_i l_i/(B\cdot\max_i l_i)$；动态 batch 满足 $\sum_i l_i\le T_{budget}$，并以 token 数而非样本数控制显存。

下面按真实案例、基线、手写机制、结果表和失败修复组织回答；所有数据都是可复现的教学实验。


## 真实案例

场景是客服与账户安全系统中的六条脱敏离线事件。字段包含工单文本、有效 token 数和风险标签；它们模拟真实的数据结构，但样本极小，只用于观察公式和状态变化。


In [1]:
import math  # 导入数学函数以实现训练与掩码公式。
import warnings  # 导入警告控制模块保持输出干净。
warnings.filterwarnings('ignore', message='The pynvml package is deprecated')  # 屏蔽环境依赖的非教学弃用提示。
import torch  # 导入张量计算和自动微分能力。
import torch.nn as nn  # 导入模块基类以手写网络结构。
torch.manual_seed(29)  # 固定随机种子使教学输出可复现。
torch.set_num_threads(1)  # 限制小实验 CPU 线程数。
samples = [  # 构造六条脱敏客服对话作为真实语义样本。
    {'id': 'C01', 'text': '支付重复扣款，申请退款', 'tokens': 6, 'risk': 1},  # 资金风险工单。
    {'id': 'C02', 'text': '收不到登录验证码', 'tokens': 2, 'risk': 0},  # 登录支持工单。
    {'id': 'C03', 'text': '账户有陌生转账记录', 'tokens': 5, 'risk': 1},  # 账户安全工单。
    {'id': 'C04', 'text': '修改订单收货地址', 'tokens': 3, 'risk': 0},  # 售后咨询工单。
    {'id': 'C05', 'text': '银行卡盗刷需要冻结', 'tokens': 7, 'risk': 1},  # 高优先级安全工单。
    {'id': 'C06', 'text': '更正发票抬头信息', 'tokens': 4, 'risk': 0},  # 账单服务工单。
]  # 结束教学数据定义。
features = torch.tensor([[1.0, 0.0, 1.0], [0.0, 1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 1.0], [1.0, 1.0, 0.0], [0.0, 1.0, 1.0]])  # 构造三维可解释特征。
labels = torch.tensor([1, 0, 1, 0, 1, 0])  # 构造风险分类标签。
print('教学实验：六条脱敏离线客服事件，只验证机制，不代表线上收益。')  # 声明数据边界。
for row in samples:  # 逐条展示真实语义输入。
    print(f"{row['id']} | token={row['tokens']} | risk={row['risk']} | {row['text']}")  # 输出样本字段。
print(f'特征形状={tuple(features.shape)}，标签={labels.tolist()}')  # 输出张量形状。


教学实验：六条脱敏离线客服事件，只验证机制，不代表线上收益。
C01 | token=6 | risk=1 | 支付重复扣款，申请退款
C02 | token=2 | risk=0 | 收不到登录验证码
C03 | token=5 | risk=1 | 账户有陌生转账记录
C04 | token=3 | risk=0 | 修改订单收货地址
C05 | token=7 | risk=1 | 银行卡盗刷需要冻结
C06 | token=4 | risk=0 | 更正发票抬头信息
特征形状=(6, 3)，标签=[1, 0, 1, 0, 1, 0]


## Baseline / 基线

先在同一批六条事件上运行最简单方案。基线不是稻草人，它提供固定的输入、口径和可比较指标。


In [2]:
request_lengths = [6, 2, 5, 3, 7, 4]  # 记录六条真实客服请求的有效 token 长度。
static_batches = [request_lengths[:3], request_lengths[3:]]  # 按固定三条请求切分 batch。
static_slots = sum(len(batch) * max(batch) for batch in static_batches)  # 统计固定 batch 需要分配的 padded token 槽位。
used_tokens = sum(request_lengths)  # 统计真实有效 token 总数。
baseline_metric = used_tokens / static_slots  # 计算固定条数 batch 的 token 利用率。
print(f'固定 3 条一批：batch={static_batches}，有效 token={used_tokens}，分配槽位={static_slots}，利用率={baseline_metric:.2%}')  # 输出基线 padding 浪费。


固定 3 条一批：batch=[[6, 2, 5], [3, 7, 4]]，有效 token=27，分配槽位=39，利用率=69.23%


## 手写核心实现与中间量

代码保留关键分子分母、mask、梯度、参数组或重算路径，而不让 Trainer 或高层框架隐藏面试问题本身。


In [3]:
budget = 10  # 设置单个动态 microbatch 的 token 上限。
sorted_lengths = sorted(enumerate(request_lengths), key=lambda pair: pair[1], reverse=True)  # 按长度从大到小排列请求以减少碎片。
dynamic_batches = []  # 保存每个动态 batch 的请求索引与 token 数。
for index, length in sorted_lengths:  # 逐个执行贪心装箱。
    placed = False  # 标记当前请求是否已经放入现有 batch。
    for batch in dynamic_batches:  # 尝试放入已经打开的 batch。
        if batch['tokens'] + length <= budget:  # 检查加入后是否仍满足 token budget。
            batch['items'].append(index)  # 记录本请求索引。
            batch['tokens'] += length  # 累加 batch 的有效 token。
            placed = True  # 标记为已放置。
            break  # 停止搜索其他 batch。
    if not placed:  # 当现有 batch 都放不下时。
        dynamic_batches.append({'items': [index], 'tokens': length})  # 新建一个 batch。
allocated_tokens = len(dynamic_batches) * budget  # 以 token budget 估算为这些 batch 预留的容量。
core_metric = used_tokens / allocated_tokens  # 计算动态 token budget 利用率。
print(f'动态 token budget={budget}：{dynamic_batches}，batch 数={len(dynamic_batches)}，利用率={core_metric:.2%}')  # 展示手写装箱结果。


动态 token budget=10：[{'items': [4, 3], 'tokens': 10}, {'items': [0, 5], 'tokens': 10}, {'items': [2, 1], 'tokens': 7}]，batch 数=3，利用率=90.00%


In [4]:
comparison_rows = [('Baseline', float(baseline_metric)), ('核心机制', float(core_metric))]  # 建立同一口径的结果表。
for name, metric in comparison_rows:  # 逐行输出结果。
    print(f'{name:<8} | 指标={metric:.6f}')  # 展示可读数值对照。


Baseline | 指标=0.692308
核心机制     | 指标=0.900000


## 结果解读

基线和核心输出只在本受控案例中比较。生产调度要处理样本随机性、DDP rank 对齐、OOM 余量和吞吐波动；严格排序会损害数据随机性，应结合 bucket shuffle。 观察结果时应关注中间量是否符合公式，而不是把六条样本上的数字宣传为线上收益。

## 失败案例

下一个单元故意破坏关键假设，并用实现修复证明该假设为何必要。


In [5]:
wrong_budget = 6  # 故意把最长请求长度误当成可忽略的上限。
dropped = [length for length in request_lengths if length > wrong_budget]  # 观察小预算是否会静默丢弃长请求。
failure_metric = len(dropped)  # 统计被错误丢弃的请求数。
fix_metric = max(request_lengths) <= budget  # 验证修复后的 budget 至少容纳最长单条请求。
print(f'失败：budget={wrong_budget} 时会丢弃长度={dropped} 的请求；修复：budget={budget}，最长请求可容纳={fix_metric}')  # 展示 OOM 余量不等于允许丢数据。


失败：budget=6 时会丢弃长度=[7] 的请求；修复：budget=10，最长请求可容纳=True


## 工程取舍、常见坑与延伸追问

**工程取舍：** 生产调度要处理样本随机性、DDP rank 对齐、OOM 余量和吞吐波动；严格排序会损害数据随机性，应结合 bucket shuffle。

**常见坑：** 把 prompt 与 label token 长度漏算，或让不同 rank 的 batch token 数差异过大导致 collective 等待。

**延伸追问：** 训练 packing 和 serving continuous batching 的共同点与差异是什么？如何根据历史 OOM 设置 token budget 安全余量？

## 生产差距

本 Notebook 在 CPU/FP32 下处理 6 条离线事件，省略了真实 token packing、分布式同步、混合精度、checkpoint、隐私治理、监控告警和灰度回滚。生产版本必须替换为受审计的数据管道与系统级指标。


In [6]:
assert len(dynamic_batches) == 3  # 验证六条请求被装入三个 token budget batch。
assert all(batch['tokens'] <= budget for batch in dynamic_batches)  # 验证每个 batch 都不超 token 上限。
assert failure_metric == 1  # 验证错误预算会丢弃最长请求。
assert fix_metric  # 验证修复预算覆盖最长请求。
